**Intelligent Vision Systems · Lab 7 · Reading & Implementing Research Papers**
Companion to **Lecture 7 — Reading & Implementing Research Papers**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/02-attention-from-scratch.ipynb) &nbsp; [Lecture 7 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/07_Paper_Implementation.html#/attention-equation) &nbsp; [All Lab 7 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-7-papers) &nbsp; [Setup guide: Colab & Cursor](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/09_paper_implementation_setup.html)

# Lab 7B · Attention from Scratch (Vaswani et al., 2017, §3.2)

| ⏱ Time | 💻 Runs on | 🎯 You will build |
|---|---|---|
| 60 min | Google Colab (CPU is enough, GPU is faster) or Cursor / VS Code on your laptop | scaled dot-product attention, masks and multi-head attention, checked against PyTorch's own implementation |

**How to work.** Run the cells from top to bottom with **Shift + Enter**. Cells marked **🖐 Your turn** have a `# TODO`:
write a few lines, run the cell, then run the **✅ check** below it. A 💡 means "not yet": read the hint, fix, run again.
Stuck for more than 10 minutes? Open the **Hint** boxes, look at the matching lecture slide, or ask the AI assistant to
*explain* (not write) the step. 

📄 **Paper:** A. Vaswani et al., *Attention Is All You Need*, NeurIPS 2017 · [arXiv:1706.03762](https://arxiv.org/abs/1706.03762) ·
[PDF](https://arxiv.org/pdf/1706.03762) · read **§3.2 (pages 3–5)** and **Figure 2** before you start.

> *"An attention function can be described as mapping a query and a set of key-value pairs to an output … The output is
> computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility
> function of the query with the corresponding key."* (§3.2)

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V \qquad\text{(Eq. 1)}$$

**In plain words:** every word asks a question (**query**), every word offers a label (**key**) and some content
(**value**). Compare the question with every label (dot product = similarity), turn the similarities into percentages
(softmax), and take that mix of the contents. Like searching YouTube: your search text is the query, video titles are
keys, the videos are values.

In [ ]:
# Setup: run this first (Colab already has PyTorch; on your laptop see requirements.txt).
# !pip install -q torch torchvision matplotlib scikit-learn opencv-python

import os, urllib.request
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_7"
for f in ["lab7_checks.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("lab files ready")

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab7_checks as C          # auto-checks + helpers for this lab

C.seed_everything(0)
device = C.get_device()
print("PyTorch", torch.__version__, "· device:", device)

## 1 · Attention by hand: 3 words, 2 numbers each

Before any code, a tiny example you can check with a pencil. Word 0's query is most similar to word 2's key, so word 0
should mostly "look at" word 2.

In [ ]:
Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
K = torch.tensor([[0., 1.], [1., 0.], [2., 0.]])
V = torch.tensor([[10., 0.], [0., 10.], [5., 5.]])
scores = Q @ K.T                              # (3, 3): row i = how much query i matches each key
print("scores =\n", scores)
print("scaled by sqrt(d_k) = sqrt(2):\n", scores / math.sqrt(2))
w = (scores / math.sqrt(2)).softmax(-1)
print("weights (rows sum to 1):\n", w.round(decimals=3))
print("output = weights @ V:\n", (w @ V).round(decimals=2))

**Check yourself (pencil):** query 0 = (1, 0) gives scores (0, 1, 2) → divided by √2: (0, 0.71, 1.41) → softmax
≈ [0.14, 0.28, 0.58]. Word 0 takes 58 % of its output from word 2, so its output ≈ 0.14·(10, 0) + 0.28·(0, 10) + 0.58·(5, 5)
= (4.3, 5.7).

## 2 · A numerically stable softmax

$\mathrm{softmax}(x)_i = e^{x_i} / \sum_j e^{x_j}$. But $e^{1000}$ overflows to `inf`. The trick every library uses:
subtract the maximum first. It gives the same answer because $e^{x-m}/\sum e^{x-m} = e^{x}/\sum e^{x}$.

### 🖐 Your turn 1: `softmax(x)` over the last dimension
<details><summary><b>Hint 1</b></summary>

`m = x.max(dim=-1, keepdim=True).values`

</details>
<details><summary><b>Hint 2</b></summary>

`e = (x - m).exp()` then divide by `e.sum(dim=-1, keepdim=True)`.

</details>

In [ ]:
def softmax(x):
    # TODO: stable softmax over the last dimension (do not call torch.softmax)
    return x

In [ ]:
C.check_softmax(softmax)

## 3 · Scaled dot-product attention (Eq. 1, Figure 2 left)

Follow Figure 2 from the bottom up: **MatMul → Scale → Mask (opt.) → SoftMax → MatMul**.
Shapes: `q` (…, L_q, d_k), `k` (…, L_k, d_k), `v` (…, L_k, d_v) → output (…, L_q, d_v). The "…" can be batch and heads.

`mask` (optional) is a **bool** tensor: `True` = may attend, `False` = blocked. Blocked scores become −∞, so after the
softmax their weight is exactly 0.

### 🖐 Your turn 2: `attention(q, k, v, mask=None)` → `(output, weights)`
<details><summary><b>Hint 1</b></summary>

Transpose only the last two axes of k: `k.transpose(-2, -1)`. (`k.T` breaks for 3-D and 4-D tensors.)

</details>
<details><summary><b>Hint 2</b></summary>

Scale: `/ math.sqrt(q.size(-1))`.

</details>
<details><summary><b>Hint 3</b></summary>

Mask: `scores = scores.masked_fill(~mask, float('-inf'))`, BEFORE the softmax.

</details>

### 📄 ↔ 💻 Paper ↔ Code: Eq. (1) and Figure 2 (left) → `attention(q, k, v, mask)`

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.2.1, Eq. (1), Figure 2</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_eq1.png" width="100%" alt="pc_t_eq1.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_transformer_fig2.png" width="100%" alt="p_transformer_fig2.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>def attention(q, k, v, mask=None):
    d_k = q.size(-1)
    # Figure 2 (left), read bottom to top: one line of code per box
    # [MatMul]   scores = Q K^T           (B,Lq,d_k) @ (B,d_k,Lk) -&gt; (B,Lq,Lk)
    # [Scale]    divide by sqrt(d_k)
    # [Mask]     optional: -inf where mask is False
    # [SoftMax]  over the keys = the LAST dim (rows sum to 1)
    # [MatMul]   weights @ V              (B,Lq,Lk) @ (B,Lk,d_v) -&gt; (B,Lq,d_v)
    return output, weights</pre>
<b>Shapes:</b> q (B, L<sub>q</sub>, d<sub>k</sub>), k (B, L<sub>k</sub>, d<sub>k</sub>), v (B, L<sub>k</sub>, d<sub>v</sub>) → output (B, L<sub>q</sub>, d<sub>v</sub>), weights (B, L<sub>q</sub>, L<sub>k</sub>)
</td></tr>
</table>

In [ ]:
def attention(q, k, v, mask=None):
    # TODO 1: scores = Q K^T / sqrt(d_k)
    # TODO 2: if mask is given, put -inf where mask is False
    # TODO 3: weights = softmax over the keys
    # TODO 4: output = weights @ V
    return v, None

In [ ]:
C.check_attention(attention)
C.check_attention_mask(attention)

## 4 · Why divide by $\sqrt{d_k}$? An experiment

The paper's footnote 4 says: if the entries of $q$ and $k$ have variance 1, their dot product has variance $d_k$.
Big numbers push the softmax into a corner where one weight is ≈ 1 and the gradients are ≈ 0. Let's see it.

### 📄 ↔ 💻 Paper ↔ Code: why the paper divides by √dₖ (§3.2.1 and footnote 4)

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.2.1, footnote 4</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_sqrtdk.png" width="100%" alt="pc_t_sqrtdk.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_fn4.png" width="100%" alt="pc_t_fn4.png"><br></td>
<td width="50%" valign="top">
<b>💻 Code</b>
<pre># footnote 4: q·k = sum of d_k products, mean 0, variance d_k
for d_k in [4, 64, 512]:
    q, k = torch.randn(10_000, d_k), torch.randn(10_000, d_k)
    dots = (q * k).sum(-1)
    print(d_k, round(dots.var().item(), 1),
          round((dots / d_k ** 0.5).var().item(), 2))
# variance ~ d_k without scaling, ~ 1 after dividing by sqrt(d_k)
# large scores -&gt; softmax ~ one-hot -&gt; &quot;extremely small gradients&quot;</pre>
</td></tr>
</table>

In [ ]:
torch.manual_seed(0)
dims = [4, 16, 64, 256, 1024]
for name, scale in [("without scaling", False), ("with 1/sqrt(d_k)", True)]:
    maxw = []
    for d in dims:
        q, k = torch.randn(1000, 1, d), torch.randn(1000, 10, d)
        s = (q @ k.transpose(-2, -1)).squeeze(1)
        if scale:
            s = s / math.sqrt(d)
        maxw.append(s.softmax(-1).max(-1).values.mean().item())
    plt.plot(dims, maxw, "o-", label=name)
    print(f"{name:18s} std of scores at d_k=1024: {s.std():.1f} · mean largest weight:", [round(m, 2) for m in maxw])
plt.xscale("log"); plt.xlabel("d_k"); plt.ylabel("mean of the largest softmax weight (10 keys)")
plt.axhline(0.1, ls=":", c="gray"); plt.legend(); plt.title("Unscaled scores saturate the softmax"); plt.show()

**Read the plot:** without scaling, at $d_k$ = 1024 one key gets almost all the weight, whatever the content: attention
stops learning. With the paper's $1/\sqrt{d_k}$ the softmax stays soft (10 keys → about 0.1–0.3 each).

## 5 · The causal (look-ahead) mask

In the decoder, word *i* must not see words after *i* (it would be cheating at training time: the answer is right
there). The paper: *"We … modify the self-attention sub-layer in the decoder stack to prevent positions from attending to
subsequent positions"* (§3.1).

### 🖐 Your turn 3: `causal_mask(L)` → bool (L, L), True where attention is allowed
<details><summary><b>Hint 1</b></summary>

`torch.ones(L, L, dtype=torch.bool)` and `torch.tril(...)`.

</details>
<details><summary><b>Hint 2</b></summary>

Row i, column j is allowed when j ≤ i (the diagonal too).

</details>

### 📄 ↔ 💻 Paper ↔ Code: masking in the decoder (§3.2.3) → `causal_mask(L)`

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.2.3</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_mask.png" width="100%" alt="pc_t_mask.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>def causal_mask(L):
    # &quot;attend to all positions ... up to and including that position&quot;
    # -&gt; row i may see columns 0 .. i : a lower-triangular bool (L, L)
    ...
&nbsp;
# inside attention(): &quot;masking out (setting to -inf) ... the input of the softmax&quot;
scores = scores.masked_fill(~mask, float(&quot;-inf&quot;))     # BEFORE the softmax</pre>
</td></tr>
</table>

In [ ]:
def causal_mask(L):
    # TODO
    return torch.ones(L, L, dtype=torch.bool)

In [ ]:
C.check_causal_mask(causal_mask)
print(causal_mask(5).int())

## 6 · Multi-head attention (§3.2.2, Figure 2 right)

$$\mathrm{MultiHead}(Q,K,V) = \mathrm{Concat}(\mathrm{head}_1,\dots,\mathrm{head}_h)\,W^O,\quad
\mathrm{head}_i = \mathrm{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

Instead of one big attention, $h$ small ones look for different things (one head for "who is the subject?", another
for "which word is nearby?"). The efficient way: one `nn.Linear(d_model, d_model)` for all heads, then **split** the
result into $h$ pieces of size $d_k = d_{model}/h$:

```
(B, L, d_model) --view--> (B, L, h, d_k) --transpose(1, 2)--> (B, h, L, d_k)   # heads become a batch axis
attention(...)  ->  (B, h, L, d_k) --transpose(1, 2)--> (B, L, h, d_k) --reshape--> (B, L, d_model) --W_o--> out
```

### 🖐 Your turn 4: complete `forward`
Keep the attribute names `w_q, w_k, w_v, w_o`: the check copies PyTorch's weights into them and compares the outputs
with `torch.nn.MultiheadAttention`.
<details><summary><b>Hint 1</b></summary>

`B, Lq, _ = q.shape` and `Lk = k.size(1)` (keys may be longer than queries in cross-attention).

</details>
<details><summary><b>Hint 2</b></summary>

`q = self.w_q(q).view(B, Lq, self.h, self.d_k).transpose(1, 2)`, same for k and v with Lk.

</details>
<details><summary><b>Hint 3</b></summary>

After attention: `out.transpose(1, 2).reshape(B, Lq, self.h * self.d_k)`, then `self.w_o(out)`.

</details>

### 📄 ↔ 💻 Paper ↔ Code: MultiHead and Figure 2 (right) → `MultiHeadAttention`

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.2.2, Figure 2 (right)</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_mha.png" width="100%" alt="pc_t_mha.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_transformer_fig2.png" width="100%" alt="p_transformer_fig2.png"><br></td>
<td width="50%" valign="top">
<b>💻 Your implementation plan: one line per box / term of the paper</b>
<pre>class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        # W_i^Q, W_i^K, W_i^V of ALL heads: one nn.Linear(d_model, d_model) each
        # W^O: nn.Linear(d_model, d_model);  d_k = d_v = d_model / h  (512/8 = 64)
    def forward(self, q, k, v, mask=None):
        # 1 [Linear]      w_q(q), w_k(k), w_v(v)              (B, L, d_model)
        # 2 split heads:  .view(B, L, h, d_k).transpose(1, 2) (B, h, L, d_k)
        # 3 [Scaled Dot-Product Attention] for all heads      (B, h, Lq, d_k)
        # 4 [Concat]:     .transpose(1, 2).reshape(B, Lq, h * d_k)
        # 5 [Linear] W^O: w_o(...)                            (B, Lq, d_model)</pre>
<b>Shapes:</b> (B, L, 512) → view → (B, L, 8, 64) → transpose → (B, 8, L, 64) → attention → back to (B, L, 512)
</td></tr>
</table>

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.h, self.d_k = n_heads, d_model // n_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.attn = None            # we keep the last attention weights for plotting

    def forward(self, q, k, v, mask=None):
        # TODO 1: project q, k, v and split into heads -> (B, h, L, d_k)
        # TODO 2: out, self.attn = attention(q, k, v, mask)
        # TODO 3: merge the heads back -> (B, Lq, d_model) and apply w_o
        return q

In [ ]:
C.check_mha(MultiHeadAttention)

## 7 · See attention work: which word looks at which?

We give 6 words hand-made 4-number embeddings: `cat` and `mat` share a feature, `sat` and `on` share another, `the`
is a bit like everything. With Q = K = V = the embeddings (self-attention with no learned weights) words attend to
similar words.

In [ ]:
words = ["the", "cat", "sat", "on", "the", "mat"]
E = torch.tensor([[0.3, 0.3, 0.3, 0.3],   # the
                  [2.0, 0.0, 0.0, 0.5],   # cat
                  [0.0, 2.0, 0.0, 0.0],   # sat
                  [0.0, 1.5, 0.5, 0.0],   # on
                  [0.3, 0.3, 0.3, 0.3],   # the
                  [2.0, 0.0, 0.0, -0.5]]) # mat
out, w = attention(E, E, E)
_, wc = attention(E, E, E, causal_mask(6))
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, m, t in [(ax[0], w, "self-attention"), (ax[1], wc, "with causal mask (decoder)")]:
    a.imshow(m, cmap="Blues", vmin=0, vmax=1)
    a.set_xticks(range(6), words); a.set_yticks(range(6), words); a.set_xlabel("key (looked at)"); a.set_ylabel("query")
    for i in range(6):
        for j in range(6):
            a.text(j, i, f"{m[i, j]:.2f}", ha="center", va="center", fontsize=8, color="k" if m[i, j] < 0.6 else "w")
    a.set_title(t)
plt.tight_layout(); plt.show()

## ✍️ Your turn (answer in a markdown cell)
1. In the left plot, which word does `cat` attend to most, apart from itself? Why?
2. In the right plot, why is the first row `[1, 0, 0, 0, 0, 0]`?
3. With d_model = 512 and h = 8 (the paper's base model), what is $d_k$? How many parameters does one multi-head attention layer have? (Count 4 linear layers with biases; then check with `C.count_params(MultiHeadAttention(512, 8))`.)
4. Change `E` so that `on` attends mostly to `mat`. What did you change?

**Next:** [Lab 7C · Build the whole Transformer and teach it to reverse sequences](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/03-transformer-reverse-sequence.ipynb)